## 상품 CRUD - products

> 공유용 사본: 인증정보와 실행 결과를 포함하지 않습니다. 실행 시 Supabase 클라이언트와 로그인 세션은 공유받는 사람이 별도로 설정해야 합니다.

In [ ]:
from supabase_client import get_supabase_client

supabase = get_supabase_client()

In [ ]:
import datetime
from zoneinfo import ZoneInfo
datetime.datetime.now(ZoneInfo("Asia/Seoul"))


class product_service:
    def __init__(self, supabase):
        user = supabase.auth.get_user()

        if not user or not user.user:
            raise PermissionError("로그인이 필요합니다.")

        print("정상 로그인 되었습니다.")

        self.supabase = supabase
        self.user = user.user

    # 상품 추가
    def add_product(self, name, price):
        response = (
            self.supabase.table("product")
                .insert({
                    "name": name,
                    "price": price,
                    "seller_id": self.user.id
                })
                .execute()
        )

        return response.data

    # 상품 조회
    def get_product(self, product_id):
        response = (
            self.supabase.table("prouduct")
                .select("*")
                .eq("id", product_id)
                .execute()

        )
        return response.data
    # 셀러 상품 조회
    def get_seller_product(self):
        response = (
            self.supabase.table("prouduct")
                .select("*")
                .eq("seller_id", self.user.id)
                .is_("deleted_at","null")
                .execute()

        )
        return response.data

    # 상품 수정
    def set_product(self, product_id, name=None, price=None):
        if name is None and price is None:
            return

        up_data = {}
        

        if not name.strip():
            up_data["name"] = name

        if not price:
            up_data["price"] = price

        up_data["modified_at"] = datetime.datetime.now(
            ZoneInfo("Asia/Seoul")
        ).isoformat()

        print(up_data)

        response = (
            self.supabase.table("product")
                .update(up_data)
                .eq("id", product_id)
                .eq("seller_id", self.user.id)
                .is_("deleted_at", "null")
                .execute()
        )

        return response.data

    # 상품 소프트 삭제
    def remove_product(self, product_id):
        deleted_at = datetime.datetime.now(
            ZoneInfo("Asia/Seoul")
        ).isoformat()

        response = (
            self.supabase.table("product")
                .update({
                    "deleted_at": deleted_at
                })
                .eq("id", product_id)
                .eq("seller_id", self.user.id)
                .is_("deleted_at", "null")
                .execute()
        )

        return response.data